In [18]:
"""
fetch_mt5_history.py
--------------------------------
Pulls historical OHLCV data from MetaTrader 5
for multiple symbols and stores them as CSVs
and pandas DataFrames (for backtesting).

Requires: MetaTrader5, pandas
"""

import MetaTrader5 as mt5
import pandas as pd
from datetime import datetime, timedelta
import os
import logging

# ---------------------- Configuration ----------------------
SAVE_PATH = "mt5_history_data"
TIMEFRAME = mt5.TIMEFRAME_M1      # 1-minute bars (best for resampling)
LOOKBACK_DAYS = 90                # Number of days of history to download
SYMBOLS = [
    "EURUSD", "GBPUSD", "USDJPY", "USDCHF", "AUDUSD",
    "USDCAD", "NZDUSD", "EURGBP", "EURJPY", "EURCHF",
    "EURAUD", "GBPJPY", "GBPCHF", "GBPAUD", "GBPCAD",
    "GBPNZD", "AUDJPY", "AUDNZD", "AUDCAD", "AUDCHF",
    "CADJPY", "CHFJPY", "NZDJPY", "NZDCAD", "NZDCHF",
    "XAUUSD", "XAGUSD", "EURCAD"
]
# -----------------------------------------------------------

def initialize_mt5():
    if not mt5.initialize():
        raise RuntimeError("❌ MT5 initialization failed")
    logging.info(f"Connected to MT5 build {mt5.version()}")

def fetch_symbol_data(symbol: str, timeframe=TIMEFRAME, days=LOOKBACK_DAYS) -> pd.DataFrame:
    """Fetch OHLCV data for a given symbol and timeframe."""
    end = datetime.now()
    start = end - timedelta(days=days)
    rates = mt5.copy_rates_range(symbol, timeframe, start, end)
    if rates is None or len(rates) == 0:
        logging.warning(f"No data for {symbol}")
        return pd.DataFrame()
    df = pd.DataFrame(rates)
    df['time'] = pd.to_datetime(df['time'], unit='s')
    df = df.rename(columns={'tick_volume': 'volume'})
    df.set_index('time', inplace=True)
    return df[['open', 'high', 'low', 'close', 'volume']]

def fetch_all_symbols(symbols=SYMBOLS, save_path=SAVE_PATH) -> dict:
    """Fetch all symbols and save each as CSV; returns dict of DataFrames."""
    os.makedirs(save_path, exist_ok=True)
    all_data = {}
    for sym in symbols:
        print(f"Fetching {sym} ...")
        df = fetch_symbol_data(sym)
        if df.empty:
            continue
        file_path = os.path.join(save_path, f"{sym}.csv")
        df.to_csv(file_path)
        all_data[sym] = df
        print(f"✅ Saved {sym} ({len(df)} rows) → {file_path}")
    return all_data

def main():
    logging.basicConfig(level=logging.INFO, format="%(asctime)s - %(message)s")
    initialize_mt5()
    data = fetch_all_symbols(SYMBOLS)
    mt5.shutdown()
    print("All historical data successfully downloaded.")
    return data

# if __name__ == "__main__":
#     main()


In [56]:
# import pandas as pd
# import numpy as np
# import talib
# from scipy.signal import argrelextrema
# from typing import Dict, Any, List
# import itertools

# class StrategyBacktester:
#     """
#     Backtester for the MT5AdvancedTradingBot strategy.
#     - Expects per-symbol minute-level OHLCV DataFrames with columns: ['open','high','low','close','volume']
#     - Resamples to H1 (trend) and M5 (entries).
#     - Simulates pending orders placed at swing +/- pip_distance and execution on subsequent bars.
#     - Implements RR-based trailing SL (break-even at 1.0R, +0.5R at 1.5R, +1.0R at 2.0R, ...).
#     """

#     def __init__(self, symbol_data: Dict[str, pd.DataFrame], initial_balance: float = 10000.0):
#         """
#         symbol_data: dict of symbol -> minute OHLCV DataFrame indexed by datetime
#         """
#         self.symbol_data = symbol_data
#         self.initial_balance = initial_balance
#         self.results = {}

#     @staticmethod
#     def detect_pip_value(df: pd.DataFrame) -> float:
#         """Heuristic to detect pip size based on price magnitude."""
#         sample = df['close'].dropna().iloc[-100:]
#         med = sample.median()
#         if med > 100: 
#             return 0.01
#         if med > 10:
#             return 0.1 if med > 50 else 0.01
#         return 0.0001

#     @staticmethod
#     def resample_to(df: pd.DataFrame, timeframe: str) -> pd.DataFrame:
#         """Resample minute data to the given timeframe ('5T', '60T')."""
#         agg = {'open': 'first', 'high': 'max', 'low': 'min', 'close': 'last', 'volume': 'sum'}
#         return df.resample(timeframe).agg(agg).dropna()

#     @staticmethod
#     def calculate_indicators(df: pd.DataFrame) -> pd.DataFrame:
#         df = df.copy()
#         df['EMA21'] = talib.EMA(df['close'], 21)
#         df['EMA13'] = talib.EMA(df['close'], 13)
#         df['EMA8'] = talib.EMA(df['close'], 8)
#         typical_price = (df['high'] + df['low'] + df['close']) / 3
#         df['VWAP'] = (typical_price * df['volume']).cumsum() / (df['volume'].cumsum() + 1e-9)
#         df['RSI'] = talib.RSI(df['close'], 14)
#         df['ROC'] = talib.ROC(df['close'], 3)
#         df['ATR'] = talib.ATR(df['high'], df['low'], df['close'], timeperiod=5)
#         return df.dropna()

#     @staticmethod
#     def get_last_swing(df: pd.DataFrame, side='high', order=5):
#         arr = df[side].values
#         extrema = argrelextrema(arr, np.greater if side == 'high' else np.less, order=order)[0]
#         if len(extrema) == 0:
#             return None, None
#         idx = extrema[-1]
#         return df.index[idx], df.iloc[idx][side]

#     def simulate_symbol(self, symbol: str, params: Dict[str, Any]) -> Dict[str, Any]:
#         """
#         Simulate trading for a single symbol.
#         params: {'atr_multiplier', 'rr_ratio', 'pip_buffer_pips', 'x_confirm_pips', 'risk_percent'}
#         """
#         df = self.symbol_data[symbol].copy()
#         df.index = pd.to_datetime(df.index)

#         # Prepare data
#         h1 = self.calculate_indicators(self.resample_to(df, '60T'))
#         m5 = self.calculate_indicators(self.resample_to(df, '5T'))
#         pip_size = self.detect_pip_value(df)

#         trades = []

#         for t in range(max(21, 5), len(m5)-1):
#             time_t = m5.index[t]
#             h1_time = h1.index[h1.index.get_indexer([time_t], method='ffill')[0]]
#             latest_h1 = h1.loc[h1_time]
#             latest_m5 = m5.iloc[t]

#             # Trend direction
#             bullish_trend = latest_h1['EMA21'] < latest_h1['EMA13'] < latest_h1['EMA8']
#             bearish_trend = latest_h1['EMA21'] > latest_h1['EMA13'] > latest_h1['EMA8']
#             bullish_m5 = latest_m5['EMA21'] < latest_m5['EMA13'] < latest_m5['EMA8']
#             bearish_m5 = latest_m5['EMA21'] > latest_m5['EMA13'] > latest_m5['EMA8']

#             signal = 'NEUTRAL'
#             if bullish_trend and bullish_m5 and latest_m5['close'] > latest_m5['VWAP'] and latest_m5['ROC'] > 0 and latest_m5['RSI'] < 70:
#                 signal = 'BUY'
#             elif bearish_trend and bearish_m5 and latest_m5['close'] < latest_m5['VWAP'] and latest_m5['ROC'] < 0 and latest_m5['RSI'] > 30:
#                 signal = 'SELL'
#             if signal == 'NEUTRAL':
#                 continue

#             # Find last swings
#             m5_slice = m5.iloc[:t+1]
#             _, swing_high = self.get_last_swing(m5_slice, 'high', order=5)
#             _, swing_low = self.get_last_swing(m5_slice, 'low', order=5)
#             if swing_high is None or swing_low is None:
#                 continue

#             # Confirm candle close beyond swing
#             last_close = m5_slice['close'].iloc[-2]
#             x_confirm = params.get('x_confirm_pips', 3) * pip_size
#             if signal == 'BUY' and (swing_high - last_close) < x_confirm:
#                 continue
#             if signal == 'SELL' and (last_close - swing_low) < x_confirm:
#                 continue

#             # Entry setup
#             pip_buffer = params.get('pip_buffer_pips', 2) * pip_size
#             rr_ratio = params.get('rr_ratio', 2)
#             if signal == 'BUY':
#                 entry, sl, tp = swing_high + pip_buffer, swing_low - pip_buffer, None
#                 tp = entry + (entry - sl) * rr_ratio
#             else:
#                 entry, sl, tp = swing_low - pip_buffer, swing_high + pip_buffer, None
#                 tp = entry - (sl - entry) * rr_ratio

#             # Simulate forward for trigger
#             executed = False
#             for j in range(t+1, min(len(m5), t+501)):
#                 bar = m5.iloc[j]
#                 if (signal == 'BUY' and bar['high'] >= entry) or (signal == 'SELL' and bar['low'] <= entry):
#                     executed, entry_idx = True, j
#                     break
#             if not executed:
#                 continue

#             # Once executed, monitor
#             entry_price = entry
#             initial_risk = abs(entry - sl)
#             for k in range(entry_idx, min(len(m5), entry_idx+1000)):
#                 bar = m5.iloc[k]
#                 high, low, close = bar['high'], bar['low'], bar['close']
#                 direction = 1 if signal == 'BUY' else -1
#                 current_profit = (close - entry_price) * direction
#                 rr = current_profit / initial_risk if initial_risk else 0

#                 # Trailing logic
#                 new_sl = None
#                 if rr >= 1.0 and rr < 1.5:
#                     new_sl = entry_price  # Break-even
#                 elif rr >= 1.5:
#                     trail_levels = int((rr - 1.0) / 0.5)
#                     new_sl = entry_price + direction * (0.5 * trail_levels * initial_risk)

#                 if new_sl is not None:
#                     if (signal == 'BUY' and new_sl > sl) or (signal == 'SELL' and new_sl < sl):
#                         sl = new_sl

#                 # SL/TP check
#                 if signal == 'BUY' and (low <= sl or high >= tp):
#                     exit_price = sl if low <= sl else tp
#                     pnl = (exit_price - entry_price)
#                     break
#                 if signal == 'SELL' and (high >= sl or low <= tp):
#                     exit_price = sl if high >= sl else tp
#                     pnl = (entry_price - exit_price)
#                     break
#             else:
#                 # If trade not closed
#                 exit_price, pnl = m5.iloc[-1]['close'], (exit_price - entry_price)

#             trades.append({
#                 'symbol': symbol, 'entry': entry_price, 'exit': exit_price,
#                 'pnl_pips': pnl / pip_size, 'rr': pnl / initial_risk if initial_risk else 0
#             })

#         # Results
#         df_trades = pd.DataFrame(trades)
#         return {
#             'symbol': symbol,
#             'total_trades': len(df_trades),
#             'net_pips': df_trades['pnl_pips'].sum(),
#             'win_rate': (df_trades['pnl_pips'] > 0).mean() if len(df_trades) else 0,
#             'avg_rr': df_trades['rr'].mean() if len(df_trades) else 0,
#             'trades': df_trades
#         }

#     def run_all(self, params: Dict[str, Any]):
#         results = {}
#         for symbol in self.symbol_data.keys():
#             results[symbol] = self.simulate_symbol(symbol, params)
#         self.results = results
#         return results

#     def grid_search(self, symbol: str, grid: Dict[str, List[Any]], max_combinations: int = 200):
#         """Grid search for best parameters per symbol."""
#         keys = list(grid.keys())
#         combos = list(itertools.product(*[grid[k] for k in keys]))[:max_combinations]
#         out = []
#         for combo in combos:
#             params = dict(zip(keys, combo))
#             res = self.simulate_symbol(symbol, params)
#             out.append(res)
#         return sorted(out, key=lambda x: x['net_pips'], reverse=True)
import pandas as pd
import numpy as np
import talib
from typing import Dict, Any


class StrategyBacktester:
    """
    Backtest class for MT5AdvancedTradingBot logic.
    Ensures only one trade per symbol, with stop order logging.
    Returns detailed per-symbol results in dictionary form.
    """

    def __init__(self, symbol_data: Dict[str, pd.DataFrame]):
        self.symbol_data = symbol_data
        self.results = {}

    @staticmethod
    def detect_pip_value(df: pd.DataFrame) -> float:
        """Infer pip value dynamically from symbol scale."""
        median_price = df['close'].dropna().median()
        if median_price > 100:
            return 0.01
        elif median_price > 10:
            return 0.1 if median_price > 50 else 0.01
        return 0.0001

    @staticmethod
    def calculate_indicators(df: pd.DataFrame) -> pd.DataFrame:
        """Compute EMAs, VWAP, RSI, ROC, ATR, and swing points."""
        df = df.copy()
        df['EMA21'] = talib.EMA(df['close'], 21)
        df['EMA13'] = talib.EMA(df['close'], 13)
        df['EMA8'] = talib.EMA(df['close'], 8)

        typical_price = (df['high'] + df['low'] + df['close']) / 3
        df['VWAP'] = (typical_price * df['volume']).cumsum() / (df['volume'].cumsum() + 1e-9)
        df['RSI'] = talib.RSI(df['close'], 14)
        df['ROC'] = talib.ROC(df['close'], 3)
        df['ATR'] = talib.ATR(df['high'], df['low'], df['close'], timeperiod=5)
        df['swing_high'] = df['high'].rolling(21).max()
        df['swing_low'] = df['low'].rolling(21).min()
        return df.dropna()

    def simulate_symbol(self, symbol: str, params: Dict[str, Any]) -> Dict[str, Any]:
        """Simulate trading for one symbol."""
        df = self.symbol_data[symbol].copy()
        df.index = pd.to_datetime(df.index)
        df = self.calculate_indicators(df)

        pip_value = self.detect_pip_value(df)
        rr_ratio = params.get("rr_ratio", 2)
        atr_multiplier = params.get("atr_multiplier", 2)

        trades = []
        active_trade = None
        pending_order = None  # Track stop order before activation

        for i in range(50, len(df) - 1):
            row = df.iloc[i]
            next_row = df.iloc[i + 1]
            stop_order_price = np.nan  # default if no pending order

            # ===== HANDLE PENDING STOP ORDER BEFORE TRIGGER =====
            if pending_order is not None:
                stop_order_price = pending_order['stop_order_price']
                # Check if price triggers the stop order
                if pending_order['order_type'] == 'BUY' and next_row['high'] >= stop_order_price:
                    active_trade = {
                        'open_datetime': next_row.name,
                        'open_price': stop_order_price,
                        'order_type': 'BUY',
                        'initial_sl': pending_order['initial_sl'],
                        'sl': pending_order['initial_sl'],
                        'tp': pending_order['tp'],
                        'stop_order_price': stop_order_price
                    }
                    pending_order = None  # clear after activation
                elif pending_order['order_type'] == 'SELL' and next_row['low'] <= stop_order_price:
                    active_trade = {
                        'open_datetime': next_row.name,
                        'open_price': stop_order_price,
                        'order_type': 'SELL',
                        'initial_sl': pending_order['initial_sl'],
                        'sl': pending_order['initial_sl'],
                        'tp': pending_order['tp'],
                        'stop_order_price': stop_order_price
                    }
                    pending_order = None

            # ===== IF TRADE IS ACTIVE, MANAGE IT =====
            if active_trade is not None:
                direction = 1 if active_trade['order_type'] == 'BUY' else -1
                current_price = next_row['close']
                entry_price = active_trade['open_price']
                initial_sl = active_trade['initial_sl']
                tp = active_trade['tp']
                current_sl = active_trade['sl']
                initial_risk = abs(entry_price - initial_sl)
                rr = ((current_price - entry_price) * direction) / initial_risk if initial_risk else 0

                # Trailing logic
                if rr >= 1.0 and rr < 1.5:
                    new_sl = entry_price
                elif rr >= 1.5:
                    trail_levels = int((rr - 1.0) / 0.5)
                    new_sl = entry_price + direction * (0.5 * trail_levels * initial_risk)
                else:
                    new_sl = current_sl

                if (direction == 1 and new_sl > current_sl) or (direction == -1 and new_sl < current_sl):
                    active_trade['sl'] = new_sl

                # Exit logic
                if direction == 1:
                    if next_row['low'] <= active_trade['sl']:
                        close_price = active_trade['sl']
                        status = 'CLOSED_SL'
                    elif next_row['high'] >= tp:
                        close_price = tp
                        status = 'CLOSED_TP'
                    else:
                        continue
                else:
                    if next_row['high'] >= active_trade['sl']:
                        close_price = active_trade['sl']
                        status = 'CLOSED_SL'
                    elif next_row['low'] <= tp:
                        close_price = tp
                        status = 'CLOSED_TP'
                    else:
                        continue

                trade_return = ((close_price - entry_price) * direction) / initial_risk
                pnl_pips = (close_price - entry_price) * direction * 10000

                trades.append({
                    'open_datetime': active_trade['open_datetime'],
                    'open_price': entry_price,
                    'order_type': active_trade['order_type'],
                    'close_datetime': next_row.name,
                    'close_price': close_price,
                    'returns': round(trade_return, 3),
                    'initial_sl': active_trade['initial_sl'],
                    'sl': active_trade['sl'],
                    'tp': tp,
                    'status': status,
                    'stop_order_price': active_trade['stop_order_price']
                })
                active_trade = None
                continue

            # ===== IF NO TRADE OR PENDING ORDER, CHECK FOR NEW SIGNAL =====
            if pending_order is None and active_trade is None:
                ema_up = row['EMA21'] < row['EMA13'] < row['EMA8']
                ema_down = row['EMA21'] > row['EMA13'] > row['EMA8']

                if ema_up and row['close'] > row['VWAP'] and row['ROC'] > 0 and row['RSI'] < 70:
                    signal = "BUY"
                elif ema_down and row['close'] < row['VWAP'] and row['ROC'] < 0 and row['RSI'] > 30:
                    signal = "SELL"
                else:
                    continue

                atr = row['ATR']
                if signal == "BUY":
                    stop_order_price = row['swing_high'] + 3 * pip_value
                    stop_loss = row['swing_low'] - (atr * atr_multiplier)
                    take_profit = stop_order_price + (stop_order_price - stop_loss) * rr_ratio
                else:
                    stop_order_price = row['swing_low'] - 3 * pip_value
                    stop_loss = row['swing_high'] + (atr * atr_multiplier)
                    take_profit = stop_order_price - (stop_loss - stop_order_price) * rr_ratio

                pending_order = {
                    'order_type': signal,
                    'stop_order_price': stop_order_price,
                    'initial_sl': stop_loss,
                    'tp': take_profit
                }

        # ===== RESULTS SUMMARY =====
        trades_df = pd.DataFrame(trades)
        if trades_df.empty:
            return {
                'symbol': symbol,
                'total_trades': 0,
                'net_pips': 0,
                'win_rate': 0,
                'avg_rr': 0,
                'trades': trades_df
            }

        p_cal = 10000 if "JPY" not in symbol else 100

        if "JPY" in symbol:
            p_cal = 100
        elif "XAU" in symbol:
            p_cal = 100
        else:
            p_cal = 10000
        trades_df['pnl_pips'] = trades_df.apply(
            lambda x: (x['close_price'] - x['open_price']) * (1 if x['order_type'] == 'BUY' else -1) * p_cal, axis=1)
        total_trades = len(trades_df)
        net_pips = trades_df['pnl_pips'].sum()
        win_rate = len(trades_df[trades_df['pnl_pips'] > 0]) / total_trades if total_trades else 0
        avg_rr = trades_df['returns'].mean() if total_trades else 0

        return {
            'symbol': symbol,
            'total_trades': total_trades,
            'net_pips': net_pips,
            'win_rate': win_rate,
            'avg_rr': avg_rr,
            'trades': trades_df
        }

    def run_all(self, params: Dict[str, Any]) -> Dict[str, Dict[str, Any]]:
        """Run backtest for all symbols."""
        results = {}
        for symbol in self.symbol_data.keys():
            print(f"Backtesting {symbol}...")
            results[symbol] = self.simulate_symbol(symbol, params)
        self.results = results
        return results


In [57]:

"""
save_backtest_results.py
---------------------------------
Save backtest results dictionary (from StrategyBacktester)
to a single Excel workbook with:
  - One sheet per symbol (all trades)
  - A summary sheet of performance stats
"""

import pandas as pd
import os


def save_backtest_results_to_excel(results_dict, filename="backtest_results.xlsx", save_path="."):
    """
    Save backtest results dictionary into an Excel workbook.
    
    Each symbol gets its own sheet with detailed trades.
    A 'Summary' sheet lists aggregate stats for all pairs.
    """
    # Ensure save directory exists
    os.makedirs(save_path, exist_ok=True)
    filepath = os.path.join(save_path, filename)

    # Initialize Excel writer
    with pd.ExcelWriter(filepath, engine='xlsxwriter') as writer:
        # Build summary list
        summary_data = []
        for symbol, result in results_dict.items():
            trades_df = result.get("trades", pd.DataFrame())

            # Write trades to sheet
            if not trades_df.empty:
                trades_df.to_excel(writer, sheet_name=symbol[:31], index=False)
            else:
                # Write an empty placeholder if no trades
                pd.DataFrame(columns=[
                    "open_datetime", "open_price", "order_type", "close_datetime",
                    "close_price", "returns", "initial_sl", "sl", "tp",
                    "status", "stop_order_price"
                ]).to_excel(writer, sheet_name=symbol[:31], index=False)

            summary_data.append({
                "Symbol": symbol,
                "Total Trades": result.get("total_trades", 0),
                "Net Pips": result.get("net_pips", 0),
                "Win Rate (%)": round(result.get("win_rate", 0) * 100, 2),
                "Average RR": round(result.get("avg_rr", 0), 3)
            })

        # Create summary DataFrame
        summary_df = pd.DataFrame(summary_data)
        summary_df.sort_values(by="Net Pips", ascending=False, inplace=True)
        summary_df.to_excel(writer, sheet_name="Summary", index=False)

        # Auto-adjust column width for all sheets
        for sheet_name in writer.sheets:
            worksheet = writer.sheets[sheet_name]
            for idx, col in enumerate(summary_df.columns if sheet_name == "Summary" else trades_df.columns):
                col_width = max(12, len(col) + 2)
                worksheet.set_column(idx, idx, col_width)

    print(f"✅ Backtest results saved successfully → {filepath}")




"""
load_mt5_data.py
---------------------------------
Loads previously downloaded OHLCV CSV files
from the 'mt5_history_data' folder into a
dictionary of pandas DataFrames.

Each DataFrame is indexed by datetime and has:
['open', 'high', 'low', 'close', 'volume']
"""

import pandas as pd
import os

# Folder containing your downloaded CSVs
DATA_PATH = "mt5_history_data"

def load_all_symbol_data(path=DATA_PATH) -> dict:
    """
    Loads all CSV files from the given directory
    and returns them as a dictionary of DataFrames.
    """
    if not os.path.exists(path):
        raise FileNotFoundError(f"❌ Folder '{path}' not found. Please download data first.")

    all_data = {}
    csv_files = [f for f in os.listdir(path) if f.endswith(".csv")]

    if not csv_files:
        raise FileNotFoundError(f"❌ No CSV files found in '{path}'")

    for file in csv_files:
        symbol = file.replace(".csv", "")
        file_path = os.path.join(path, file)
        df = pd.read_csv(file_path)
        df['time'] = pd.to_datetime(df['time'])
        df.set_index('time', inplace=True)
        df = df[['open', 'high', 'low', 'close', 'volume']]
        all_data[symbol] = df
        print(f"✅ Loaded {symbol}: {len(df)} rows")

    print(f"\n✅ Successfully loaded {len(all_data)} symbols.")
    return all_data


# if __name__ == "__main__":
#     data = load_all_symbol_data()
#     # Example: display one DataFrame
#     sample_symbol = list(data.keys())[0]
#     print(f"\nSample {sample_symbol} data:\n", data[sample_symbol].head())


In [60]:
# from fetch_mt5_history import main
# from strategy_backtester import StrategyBacktester

# Step 1. Pull data
data = load_all_symbol_data()  # returns {'EURUSD': DataFrame, 'GBPUSD': DataFrame, ...}

# Step 2. Backtest
bt = StrategyBacktester(data)
params = {'atr_multiplier':3, 'rr_ratio':2, 'pip_buffer_pips':2, 'x_confirm_pips':3, 'risk_percent':1}
results = bt.run_all(params)

# Save results to Excel
save_backtest_results_to_excel(results, filename="backtest_results.xlsx", save_path="backtest_outputs")



✅ Loaded AUDCAD: 92057 rows
✅ Loaded AUDCHF: 92046 rows
✅ Loaded AUDJPY: 92058 rows
✅ Loaded AUDNZD: 92048 rows
✅ Loaded AUDUSD: 92064 rows
✅ Loaded CADJPY: 92062 rows
✅ Loaded CHFJPY: 92055 rows
✅ Loaded EURAUD: 92021 rows
✅ Loaded EURCAD: 92057 rows
✅ Loaded EURCHF: 91909 rows
✅ Loaded EURGBP: 92059 rows
✅ Loaded EURJPY: 92000 rows
✅ Loaded EURUSD: 92066 rows
✅ Loaded GBPAUD: 92050 rows
✅ Loaded GBPCAD: 92039 rows
✅ Loaded GBPCHF: 92042 rows
✅ Loaded GBPJPY: 92039 rows
✅ Loaded GBPNZD: 92054 rows
✅ Loaded GBPUSD: 92064 rows
✅ Loaded NZDCAD: 92069 rows
✅ Loaded NZDCHF: 92047 rows
✅ Loaded NZDJPY: 92063 rows
✅ Loaded NZDUSD: 92069 rows
✅ Loaded USDCAD: 92069 rows
✅ Loaded USDCHF: 92060 rows
✅ Loaded USDJPY: 92070 rows
✅ Loaded XAGUSD: 87773 rows
✅ Loaded XAUUSD: 88081 rows

✅ Successfully loaded 28 symbols.
Backtesting AUDCAD...
Backtesting AUDCHF...
Backtesting AUDJPY...
Backtesting AUDNZD...
Backtesting AUDUSD...
Backtesting CADJPY...
Backtesting CHFJPY...
Backtesting EURAUD...
Backt

In [61]:
results

{'AUDCAD': {'symbol': 'AUDCAD',
  'total_trades': 41,
  'net_pips': 67.36427110967334,
  'win_rate': 0.34146341463414637,
  'avg_rr': 0.07317073170731707,
  'trades':          open_datetime  open_price order_type      close_datetime  \
  0  2025-07-28 09:21:00     0.89823       SELL 2025-07-28 16:39:00   
  1  2025-07-28 16:47:00     0.89468       SELL 2025-07-29 17:58:00   
  2  2025-07-29 23:07:00     0.89749        BUY 2025-07-30 14:56:00   
  3  2025-07-30 15:30:00     0.89377       SELL 2025-07-30 16:55:00   
  4  2025-07-30 16:57:00     0.89070       SELL 2025-07-30 21:00:00   
  5  2025-07-30 21:49:00     0.89062       SELL 2025-07-31 08:56:00   
  6  2025-07-31 11:27:00     0.89347       SELL 2025-07-31 15:50:00   
  7  2025-07-31 16:09:00     0.89024       SELL 2025-08-01 15:30:00   
  8  2025-08-07 09:28:00     0.89576        BUY 2025-08-14 18:33:00   
  9  2025-08-14 19:51:00     0.89630        BUY 2025-08-15 05:21:00   
  10 2025-08-15 06:00:00     0.89718        BUY 2025-0

In [42]:
results['AUDCAD']  #['trades'][-50:] 

{'symbol': 'AUDCAD',
 'total_trades': 41,
 'net_pips': 67.36427110967334,
 'win_rate': 0.34146341463414637,
 'avg_rr': 0.07317073170731707,
 'trades':          open_datetime  open_price order_type      close_datetime  \
 0  2025-07-28 09:21:00     0.89823       SELL 2025-07-28 16:39:00   
 1  2025-07-28 16:47:00     0.89468       SELL 2025-07-29 17:58:00   
 2  2025-07-29 23:07:00     0.89749        BUY 2025-07-30 14:56:00   
 3  2025-07-30 15:30:00     0.89377       SELL 2025-07-30 16:55:00   
 4  2025-07-30 16:57:00     0.89070       SELL 2025-07-30 21:00:00   
 5  2025-07-30 21:49:00     0.89062       SELL 2025-07-31 08:56:00   
 6  2025-07-31 11:27:00     0.89347       SELL 2025-07-31 15:50:00   
 7  2025-07-31 16:09:00     0.89024       SELL 2025-08-01 15:30:00   
 8  2025-08-07 09:28:00     0.89576        BUY 2025-08-14 18:33:00   
 9  2025-08-14 19:51:00     0.89630        BUY 2025-08-15 05:21:00   
 10 2025-08-15 06:00:00     0.89718        BUY 2025-08-15 17:59:00   
 11 2025-0

In [41]:
results.keys() 

dict_keys(['AUDCAD', 'AUDCHF'])

In [ ]:
XAUUSD
XAGUSD
EURUSD
EURAUD
EURJPY
EURGBP
CHFJPY
EURCAD
AUDJPY
AUDCAD
GBPUSD
